# Step 2: Hybrid EfficientNet-B0 + CBAM Training for 5-Stage DR Grading (APTOS 2019)

Dedicated GPU training & evaluation notebook for **Hybrid EfficientNet-B0 + CBAM (Channel & Spatial Attention)** on APTOS 2019:
- **Classes:** 5-Stage ICDR Grading (`0: No DR`, `1: Mild NPDR`, `2: Moderate NPDR`, `3: Severe NPDR`, `4: Proliferative DR`)
- **Metrics:** **Quadratic Weighted Kappa (QWK)**, Accuracy, Macro-F1, Referable DR Recall (`Grade >= 2`), & Confusion Matrix.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.models.hybrid_model import HybridDRPredictor
from src.train import RetinalFundusDataset, train_epoch, evaluate, set_seed
from src.preprocess import DR_CLASS_NAMES

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### 1. Experiment Configuration

In [ ]:
BACKBONE = 'efficientnet_b0'
USE_BEN_GRAHAM = True
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4

DATA_DIR = ROOT_DIR / 'data' / 'aptos2019'
METADATA_PATH = ROOT_DIR / 'data' / 'metadata.csv'
MODELS_DIR = ROOT_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

CKPT_NAME = f'best_dr_model_{BACKBONE}.pth'
SAVE_PATH = MODELS_DIR / CKPT_NAME
print('Checkpoint target:', SAVE_PATH)

### 2. DataLoaders & Inverse-Frequency Class Weights

In [ ]:
df = pd.read_csv(METADATA_PATH)
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

class_counts = np.maximum(np.array([(train_df['label'] == c).sum() for c in range(5)], dtype=np.float32), 1.0)
class_weights = torch.tensor(class_counts.sum() / (5.0 * class_counts), dtype=torch.float32).to(device)
print('Class Weights (Grades 0-4):', class_weights.cpu().numpy().round(3))

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(RetinalFundusDataset(train_df, DATA_DIR, train_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(RetinalFundusDataset(val_df, DATA_DIR, eval_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(RetinalFundusDataset(test_df, DATA_DIR, eval_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### 3. Train Model & Save Best QWK Checkpoint

In [ ]:
model = HybridDRPredictor(backbone=BACKBONE, num_classes=5, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_qwk, patience_cnt = -1.0, 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_qwk': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_qwk, val_f1 = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_qwk)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)

    print(f'Epoch {epoch:02d}/{EPOCHS} [{time.time()-t0:.1f}s] | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val QWK: {val_qwk:.4f} | F1: {val_f1:.4f}')
    if val_qwk > best_qwk:
        best_qwk = val_qwk
        patience_cnt = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  -> Saved best checkpoint (QWK={best_qwk:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= 5:
            print('Early stopping triggered.')
            break

### 4. Hold-Out Test Set Evaluation (QWK, Referable DR Recall & Confusion Matrix)

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
preds, targets = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        out = model(imgs.to(device))
        preds.extend(torch.argmax(out, dim=1).cpu().numpy().tolist())
        targets.extend(lbls.numpy().tolist())

preds_np, targets_np = np.array(preds), np.array(targets)
test_qwk = cohen_kappa_score(targets_np, preds_np, weights='quadratic')
ref_true = (targets_np >= 2).astype(int)
ref_pred = (preds_np >= 2).astype(int)
ref_sens = float(np.sum((ref_true == 1) & (ref_pred == 1)) / max(np.sum(ref_true == 1), 1))

print(f'Hold-Out Test Accuracy: {np.mean(preds_np == targets_np)*100:.2f}%')
print(f'Hold-Out Test QWK:      {test_qwk:.4f}')
print(f'Referable DR Recall:    {ref_sens*100:.2f}%')
print('\n' + classification_report(targets_np, preds_np, target_names=[DR_CLASS_NAMES[i] for i in range(5)]))

cm = confusion_matrix(targets_np, preds_np)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[DR_CLASS_NAMES[i] for i in range(5)], yticklabels=[DR_CLASS_NAMES[i] for i in range(5)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{BACKBONE} (Hybrid CBAM) — Confusion Matrix')
plt.tight_layout(); plt.show()